In [1]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
os.chdir(r"D:\proj-python\Videos")

# Subtract Background Manually

In [5]:
cap=cv2.VideoCapture("Vehicle Dataset Sample 4.mp4")
_,first_frame=cap.read()
cap.set(cv2.CAP_PROP_FPS,30)
first_frame=cv2.resize(first_frame,(0,0),fx=1.5,fy=1.0)
gray_first_frame=cv2.cvtColor(first_frame,cv2.COLOR_BGR2GRAY)
gray_first_frame=cv2.GaussianBlur(gray_first_frame,(5,5),0)
h,w,fps=cap.get(cv2.CAP_PROP_FRAME_HEIGHT),cap.get(cv2.CAP_PROP_FRAME_WIDTH),cap.get(cv2.CAP_PROP_FPS)
print("height",h,"width",w,"FPS",fps)
while True:
    _,frame=cap.read()
    cap.set(cv2.CAP_PROP_FPS,30)
    if frame is None:
        break
    frame=cv2.resize(frame,(0,0),fx=1.5,fy=1.0)
    gray_frame=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    gray_frame=cv2.GaussianBlur(gray_frame,(5,5),0)
    difference=cv2.absdiff(gray_first_frame,gray_frame)
    _,difference=cv2.threshold(difference,40,200,cv2.THRESH_BINARY)
    cv2.imshow("first_frame",first_frame)
    cv2.imshow("frame",frame)
    cv2.imshow("difference",difference)
    key=cv2.waitKey(30)
    if key==ord('q'):
        break
cv2.destroyAllWindows()
cap.release()

height 360.0 width 640.0 FPS 25.0


# Subtract Background MOG2

In [27]:
cap=cv2.VideoCapture("Vehicle Dataset Sample 4.mp4")
subtractor=cv2.createBackgroundSubtractorMOG2(history=100,varThreshold=60,detectShadows=False)

h,w,fps=cap.get(cv2.CAP_PROP_FRAME_HEIGHT),cap.get(cv2.CAP_PROP_FRAME_WIDTH),cap.get(cv2.CAP_PROP_FPS)
print("height",h,"width",w,"FPS",fps)

while True:
    ret,frame=cap.read()
    if not ret:
        break
    frame=cv2.resize(frame,(0,0),fx=1.5,fy=1.0)
     
    mask=subtractor.apply(frame)

    cv2.imshow("mask",mask)
    cv2.imshow("frame",frame)

    key=cv2.waitKey(30)
    if key==ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

height 360.0 width 640.0 FPS 25.0


# Advanced Threshold With MOG2

In [45]:
import cv2

cap=cv2.VideoCapture("Vehicle Dataset Sample 4.mp4")

subtractor=cv2.createBackgroundSubtractorMOG2(history=500,varThreshold=50,detectShadows=False)

kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(3,3))

while True:
    ret,frame=cap.read()
    if not ret:
        break

    frame=cv2.resize(frame,(0,0),fx=1.5,fy=1.0)

    mask=subtractor.apply(frame)

    # mask=cv2.morphologyEx(mask,cv2.MORPH_OPEN,kernel)
    mask=cv2.morphologyEx(mask,cv2.MORPH_CLOSE,kernel)
    mask=cv2.dilate(mask,kernel,iterations=1)
    # mask=cv2.erode(mask,kernel,iterations=3)

    contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        area=cv2.contourArea(contour)

        if area>500:
            x,y,w,h=cv2.boundingRect(contour)
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)

    cv2.imshow("Mask",mask)
    cv2.imshow("Vehicle Detection",frame)

    if cv2.waitKey(30)&0xFF==ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

# Vehicle Detection

In [47]:
import cv2

cap=cv2.VideoCapture("Vehicle Dataset Sample 4.mp4")

ret,first_frame=cap.read()

if not ret:
    raise RuntimeError("Could not read the video.")

first_frame=cv2.resize(first_frame,(0,0),fx=1.5,fy=1.0)
gray_first_frame=cv2.cvtColor(first_frame,cv2.COLOR_BGR2GRAY)
gray_first_frame=cv2.GaussianBlur(gray_first_frame,(5,5),0)

h,w,fps=cap.get(cv2.CAP_PROP_FRAME_HEIGHT),cap.get(cv2.CAP_PROP_FRAME_WIDTH),cap.get(cv2.CAP_PROP_FPS)
print("height",h,"width",w,"FPS",fps)

kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))

while True:
    ret,frame=cap.read()

    if not ret:
        break

    frame=cv2.resize(frame,(0,0),fx=1.5,fy=1.0)
    gray_frame=cv2.cvtColor(frame,cv2.COLOR_BGR2GRAY)
    gray_frame=cv2.GaussianBlur(gray_frame,(5,5),0)

    difference=cv2.absdiff(gray_first_frame,gray_frame)
    _,difference=cv2.threshold(difference,40,255,cv2.THRESH_BINARY)

    difference=cv2.morphologyEx(difference,cv2.MORPH_OPEN,kernel)
    difference=cv2.morphologyEx(difference,cv2.MORPH_CLOSE,kernel)
    difference=cv2.dilate(difference,kernel,iterations=1)
    contours,_=cv2.findContours(difference,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        area=cv2.contourArea(contour)

        if area>500:
            x,y,w,h=cv2.boundingRect(contour)
            cv2.rectangle(frame,(x,y),(x+w,y+h),(0,255,0),2)


    cv2.imshow("first_frame",first_frame)
    cv2.imshow("frame",frame)
    cv2.imshow("difference",difference)

    if cv2.waitKey(30)&0xFF==ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

height 360.0 width 640.0 FPS 25.0
